In [ ]:
# Global Market News Summarizer

Scrapes the latest global market news headlines from Yahoo Finance, extracts the full article content, and uses an LLM (via OpenRouter) to generate a concise, objective daily market briefing — covering overall sentiment, sectors in motion, and key investor takeaways.

**Built as a community contribution based on Day 1 of Ed Donner's LLM Engineering course.**

In [ ]:
# Import necessary libraries

import os
import requests
import time
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown,display
from openai import OpenAI

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

In [ ]:
headers = {'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'}

def fetch_website_contents(url):
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            paragraphs = soup.body.find_all("p")
            text = ""
            for p in paragraphs:
                text += p.get_text() + "\n"
        else:
            text = ""
        return (title + "\n\n" + text)[:2_000]


In [ ]:
def fetch_website_links(url):
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        all_tags = soup.find_all("a")
        links = []
        for link in all_tags:
            href = link.get("href")
            if href:
                links.append(href)
        return links

In [ ]:
raw_links = fetch_website_links("https://finance.yahoo.com/news/")
print(raw_links)


In [ ]:
article_links_raw = []

for link in raw_links:
    if "articles" in link and link.endswith(".html"):
        article_links_raw.append(link)

print(len(article_links_raw))

In [ ]:
article_links = []

for link in article_links_raw:
    if link not in article_links:
        article_links.append(link)

print(article_links)

In [ ]:
selected_links = article_links[:5]

all_articles_content = []

for link in selected_links:
        content = fetch_website_contents(link)
        all_articles_content.append(content)
        time.sleep(1)

print(len(all_articles_content))
print(all_articles_content[0])

In [ ]:
merge_articles = "\n\n---\n\n".join(all_articles_content)

In [ ]:
system_prompt = """
You are a professional market analyst who writes concise, objective daily market wrap-ups.
You will be given the combined content of several news articles about global financial markets.
Summarize them into a clear market briefing.
Do not use humor, sarcasm, or casual language — keep the tone objective and professional, like a Bloomberg or Reuters market wrap.
Respond in markdown. Do not wrap the markdown in a code block.
"""

user_prompt = f"""
Here is the combined content of several recent global market news articles:

{merge_articles}

Based on these articles, write a market briefing that includes:
1. Overall market sentiment today (bullish/bearish/mixed), with reasoning
2. Sectors or companies that are moving significantly, and why
3. 2-3 key takeaways an investor should know today

Keep it concise and skip anything unrelated to markets (ads, navigation text, unrelated topics).
"""

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

response = response.choices[0].message.content

display(Markdown(response))

In [ ]:
response = 